# NVIDIA Nemotron Reasoning Challenge - E2E Pipeline

This notebook runs the full zero-to-hero pipeline on Kaggle:
1. Load data
2. Fine-tune LoRA adapter on Nemotron-3-Nano-30B
3. Package submission.zip

**GPU Required**: RTX PRO 6000 (or similar with ≥24GB VRAM)

## 0. Setup

In [ ]:
!pip install -q peft trl datasets accelerate bitsandbytes

In [ ]:
import os
import zipfile

import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Configuration

In [ ]:
# === CONFIGURATION ===
# On Kaggle, use the model from attached input
MODEL_NAME = "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
# If running locally with HF download:
# MODEL_NAME = "nvidia/Nemotron-3-Nano-30B-A3B-BF16"

TRAIN_CSV = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv"
TEST_CSV = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/test.csv"
OUTPUT_DIR = "/kaggle/working/lora_adapter"
SUBMISSION_PATH = "/kaggle/working/submission.zip"

# LoRA config
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training config
NUM_EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 2e-4
MAX_SEQ_LEN = 4096

# Prompt
SYSTEM_PROMPT = (
    "You are a helpful assistant that solves reasoning puzzles. "
    "Think step by step. Put your final answer within \\boxed{}."
)

## 2. Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
print(f"Training examples: {len(train_df)}")
print(f"Columns: {list(train_df.columns)}")
train_df.head()

## 3. Load Model & Tokenizer

In [ ]:
# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded: {model.config._name_or_path}")
print(f"Vocab size: {len(tokenizer)}")

## 4. Apply LoRA

In [ ]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Prepare Dataset

In [ ]:
def format_chat(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": f"The answer is \\boxed{{{row['answer']}}}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}


dataset = Dataset.from_pandas(train_df)
dataset = dataset.map(format_chat, remove_columns=dataset.column_names)

# Filter sequences that are too long
dataset = dataset.filter(lambda x: len(tokenizer.encode(x["text"])) <= MAX_SEQ_LEN)
print(f"Dataset size after filtering: {len(dataset)}")
print(f"\nSample:\n{dataset[0]['text'][:500]}")

## 6. Train

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    max_grad_norm=1.0,
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()

## 7. Save & Package Submission

In [ ]:
# Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")
print(f"Files: {os.listdir(OUTPUT_DIR)}")

# Verify adapter_config.json exists
assert os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json")), (
    "Missing adapter_config.json!"
)

# Package submission.zip
with zipfile.ZipFile(SUBMISSION_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
    zf.write(config_path, "adapter_config.json")

    weights_path = os.path.join(OUTPUT_DIR, "adapter_model.safetensors")
    if not os.path.exists(weights_path):
        weights_path = os.path.join(OUTPUT_DIR, "adapter_model.bin")
    zf.write(weights_path, os.path.basename(weights_path))

size_mb = os.path.getsize(SUBMISSION_PATH) / (1024 * 1024)
print(f"\nSubmission: {SUBMISSION_PATH} ({size_mb:.1f} MB)")

## 8. Quick Sanity Check (Optional)

Run inference on a few examples to verify the adapter works.

In [ ]:
import re

model.eval()
test_df = pd.read_csv(TEST_CSV)
sample = test_df.head(3)

for _, row in sample.iterrows():
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    # Extract boxed answer
    boxed = re.findall(r"\\boxed\{([^}]+)\}", response)
    answer = boxed[-1] if boxed else "N/A"

    print(f"ID: {row['id']}")
    print(f"Response (last 200 chars): ...{response[-200:]}")
    print(f"Extracted answer: {answer}")
    print("-" * 40)

In [ ]:
print("Done! Submit submission.zip to the competition.")
print(f"File: {SUBMISSION_PATH}")